# ⚽ FPL Mini-League Differential Tracker

Run this notebook in **Google Colab** with **zero installation** on any device (phone, tablet, or laptop)!

It connects directly to the official Fantasy Premier League (FPL) API to find low-ownership, high-scoring players (differentials) for your specific mini-league (Classic or Draft).

In [ ]:
#@title 🎮 Enter League Details & Run
league_id = 19308 #@param {type:"integer"}
gameweek_id = 10 #@param {type:"integer"}
count_bench = True #@param {type:"boolean"}
top_n = 15 #@param {type:"slider", min:5, max:50, step:1}

from collections import Counter
import requests
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0"}
fpl_base = "https://fantasy.premierleague.com/api"
draft_base = "https://draft.premierleague.com/api"

print("⏳ Fetching FPL data...")
bootstrap = requests.get(f"{fpl_base}/bootstrap-static/", headers=headers).json()
player_names = {p["id"]: p["web_name"] for p in bootstrap["elements"]}

live_data = requests.get(f"{fpl_base}/event/{gameweek_id}/live/", headers=headers).json()
player_points = {p["id"]: p["stats"]["total_points"] for p in live_data["elements"]}

classic_resp = requests.get(f"{fpl_base}/leagues-classic/{league_id}/standings/?page_standings=1", headers=headers)

if classic_resp.status_code == 200:
    entries = []
    page = 1
    mode = "Classic"
    while True:
        if page > 1:
            classic_resp = requests.get(f"{fpl_base}/leagues-classic/{league_id}/standings/?page_standings={page}", headers=headers)
            if classic_resp.status_code != 200:
                break
        league_data = classic_resp.json()
        if page == 1:
            league_name = league_data["league"]["name"]
        entries.extend([
            {"entry_id": m["entry"], "team": m["entry_name"], "manager": m["player_name"]}
            for m in league_data["standings"]["results"]
        ])
        if not league_data["standings"].get("has_next"):
            break
        page += 1
    picks_template = f"{fpl_base}/entry/{{entry_id}}/event/{gameweek_id}/picks/"
else:
    draft_resp = requests.get(f"{draft_base}/league/{league_id}/details", headers=headers)
    draft_resp.raise_for_status()
    league_data = draft_resp.json()
    league_name = league_data["league"]["name"]
    entries = [
        {
            "entry_id": m["entry_id"],
            "team": m["entry_name"],
            "manager": f"{m['player_first_name']} {m['player_last_name']}",
        }
        for m in league_data["league_entries"]
    ]
    picks_template = f"{draft_base}/entry/{{entry_id}}/event/{gameweek_id}"
    mode = "Draft"

total_teams = len(entries)
ownership_counts = Counter()
team_active_picks = []

for entry in entries:
    picks_resp = requests.get(picks_template.format(entry_id=entry["entry_id"]), headers=headers)
    if picks_resp.status_code != 200:
        continue
    picks = picks_resp.json()["picks"]
    for pick in picks:
        pid = pick["element"]
        is_active = pick.get("multiplier", 1 if pick.get("position", 12) <= 11 else 0) > 0
        if count_bench or is_active:
            ownership_counts[pid] += 1
        if is_active:
            team_active_picks.append({
                "team": entry["team"],
                "manager": entry["manager"],
                "pid": pid,
                "player": player_names.get(pid, f"ID {pid}"),
                "points": player_points.get(pid, 0),
            })

candidates = []
for item in team_active_picks:
    owned_by = ownership_counts[item["pid"]]
    if item["points"] <= 0:
        continue
    item["owned_by"] = owned_by
    item["league_ownership_pct"] = round((owned_by / total_teams) * 100, 1)
    candidates.append(item)

candidates.sort(key=lambda x: (x["owned_by"], -x["points"]))

print(f"\n🏆 {league_name} ({mode} | {total_teams} Teams) — GW{gameweek_id} Differentials\n")
table_rows = []
for rank, c in enumerate(candidates[:top_n], start=1):
    table_rows.append({
        "Rank": rank,
        "Player": c["player"],
        "Points": c["points"],
        "Ownership": f"{c['owned_by']}/{total_teams} ({c['league_ownership_pct']}%)",
        "Team": c["team"],
        "Manager": c["manager"],
    })

df = pd.DataFrame(table_rows)
display(df)
